# Labwork 1 - PCA: Abalone dataset
**Họ tên:** ...  |  **MSSV:** ...

Notebook này làm Phần 1 và Phần 2 cho dataset **Abalone** (UCI id 1). Cấu trúc giống notebook Raisin; điểm khác là có feature categorical `Sex` và nhãn là `Rings` (số vòng, tuổi ≈ Rings + 1.5).

## 0. Chuẩn bị

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")
from scipy.stats import chi2_contingency, f_oneway

**Giải thích:** thêm `chi2_contingency` (kiểm định chi-square cho 2 biến categorical) và `f_oneway` (ANOVA: so sánh mean giữa các nhóm) để phân tích feature categorical ở mục 8.

## 1. Giới thiệu dataset

In [ ]:
abalone = fetch_ucirepo(id=1)
X = abalone.data.features
y = abalone.data.targets
df = pd.concat([X, y], axis=1)

df.to_csv("../data/raw/abalone_raw.csv", index=False)
print(df.shape)
df.head()

**Giải thích:** tải dữ liệu, gộp bảng, lưu bản gốc.

**Mô tả (điền thêm):** Abalone gồm 4177 mẫu bào ngư với số đo vật lý (chiều dài, đường kính, chiều cao, các loại cân nặng) và giới tính. *(TODO: viết mô tả bằng lời của bạn, trích nguồn UCI.)*

## 2. Loại của từng feature

In [ ]:
print(df.dtypes)

**TODO - điền bảng phân loại:**

| Feature | Rời rạc / Liên tục | Định lượng / Định tính | Numerical / Categorical |
|---|---|---|---|
| Sex | ? | ? | ? |
| Length | ? | ? | ? |
| Diameter | ? | ? | ? |
| Height | ? | ? | ? |
| Whole_weight | ? | ? | ? |
| Shucked_weight | ? | ? | ? |
| Viscera_weight | ? | ? | ? |
| Shell_weight | ? | ? | ? |
| Rings | ? | ? | ? |

## 3. Vấn đề chất lượng dữ liệu

In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Số dòng trùng lặp:", df.duplicated().sum())
print("Số mẫu có Height = 0:", (df["Height"] == 0).sum())
print()
print(df["Sex"].value_counts())
df.describe()

**Giải thích:** ngoài missing/duplicate, ta kiểm tra thêm `Height = 0` (chiều cao bằng 0 là không hợp lý về mặt vật lý, đây là *giá trị sai*) và các nhãn của `Sex` (M, F, I = Infant) để xem có không nhất quán không.

In [ ]:
num_cols = df.select_dtypes(include="number").columns

outlier_count = {}
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_count[col] = ((df[col] < low) | (df[col] > high)).sum()

pd.Series(outlier_count, name="so_outlier")

In [ ]:
df[num_cols].plot(kind="box", subplots=True, layout=(3, 3), figsize=(14, 9))
plt.tight_layout()
plt.savefig("../figures/abalone_boxplot_before.png", dpi=150)
plt.show()

**Giải thích:** đếm và vẽ outlier bằng quy tắc IQR.

**TODO - thảo luận:** liệt kê vấn đề tìm thấy (Height = 0, outlier ở Height và các cột cân nặng...) và ảnh hưởng của chúng đến phân tích.

## 4. Làm sạch dữ liệu

In [ ]:
df_clean = df.copy()

# 4.1 Xóa dòng trùng
df_clean = df_clean.drop_duplicates()

# 4.2 Xóa dòng thiếu dữ liệu (nếu có)
df_clean = df_clean.dropna()

# 4.3 Xóa mẫu có Height = 0 (giá trị sai)
df_clean = df_clean[df_clean["Height"] > 0]

# 4.4 Giới hạn (clip) outlier của các cột số, trừ Rings
feature_cols = [c for c in num_cols if c != "Rings"]
for col in feature_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    df_clean[col] = df_clean[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print("Trước:", df.shape, "| Sau:", df_clean.shape)
df_clean.to_csv("../data/processed/abalone_clean.csv", index=False)

**Giải thích:** mẫu `Height = 0` bị xóa vì là dữ liệu sai; outlier chỉ bị clip để giữ nhiều mẫu. Ta không clip `Rings` vì đó là nhãn.

**TODO:** so sánh trước/sau và biện minh cho lựa chọn.

In [ ]:
comparison = pd.DataFrame({
    "mean_before": df[feature_cols].mean(),
    "mean_after": df_clean[feature_cols].mean(),
    "std_before": df[feature_cols].std(),
    "std_after": df_clean[feature_cols].std(),
})
comparison

## 5. Data integration
**TODO:** Abalone là một bảng đơn, không có nguồn khác đi kèm. Viết vài câu giải thích vì sao không cần tích hợp (hoặc mô tả nếu bạn có ghép thêm nguồn).

## 6. Data transformation

In [ ]:
# Mã hóa categorical Sex bằng one-hot
df_enc = pd.get_dummies(df_clean, columns=["Sex"], dtype=int)

# Chuẩn hóa các feature số (không gồm Rings)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[feature_cols])

print(df_enc.head())

**Giải thích:** `Sex` (M/F/I) là chữ nên được *one-hot encode* thành 3 cột 0/1 (không có thứ tự nên không dùng số 0,1,2). Các feature số được standardize.

**Lưu ý cho PCA:** để đơn giản, PCA ở dưới chỉ dùng 7 feature số; `Sex` dùng để tô màu biểu đồ. **TODO:** giải thích lựa chọn này (PCA dựa trên phương sai, không phù hợp trực tiếp với biến one-hot).

## 7. Label và loại bài toán
**TODO:** nhãn là `Rings` (số vòng trên vỏ, ước lượng tuổi = Rings + 1.5). Có thể làm **hồi quy** (dự đoán Rings) hoặc **phân loại** (nhóm tuổi). Viết thêm ý nghĩa thực tế.

In [ ]:
df_clean["Rings"].describe()

## 8. Mean, variance, covariance, correlation

In [ ]:
features = df_clean[feature_cols]

stats = pd.DataFrame({"mean": features.mean(), "variance": features.var()})
print(stats)

print("\nCovariance matrix:")
print(features.cov().round(4))

corr = features.corr()

**Giải thích:** như notebook Raisin: mean, variance, covariance, correlation cho các feature số.

In [ ]:
# Phân tích feature categorical Sex

# (a) Sex vs nhóm tuổi (2 biến categorical) -> chi-square
age_group = pd.cut(df_clean["Rings"], bins=[0, 8, 10, 30], labels=["young", "middle", "old"])
table = pd.crosstab(df_clean["Sex"], age_group)
chi2, p, _, _ = chi2_contingency(table)
print(table)
print(f"Chi-square p-value = {p:.4g}")

# (b) Sex vs Rings (categorical vs numeric) -> ANOVA
groups = [g["Rings"].values for _, g in df_clean.groupby("Sex")]
f_stat, p_anova = f_oneway(*groups)
print(f"ANOVA p-value = {p_anova:.4g}")

sns.boxplot(data=df_clean, x="Sex", y="Rings")
plt.savefig("../figures/abalone_sex_rings.png", dpi=150)
plt.show()

**Giải thích:** với biến categorical dùng *bảng tần suất + chi-square* (quan hệ với biến categorical khác) và *ANOVA/boxplot* (quan hệ với biến số). p-value nhỏ (< 0.05) nghĩa là có quan hệ có ý nghĩa thống kê.

**TODO:** nhận xét kết quả (ví dụ nhóm Infant có Rings thấp hơn rõ rệt).

## 9. Cặp feature tương quan mạnh nhất

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Abalone - correlation matrix")
plt.tight_layout()
plt.savefig("../figures/abalone_corr_heatmap.png", dpi=150)
plt.show()

pairs = corr.abs().where(~np.eye(len(corr), dtype=bool)).stack()
best_pair = pairs.idxmax()
print("Cặp mạnh nhất:", best_pair, "-> r =", round(corr.loc[best_pair], 3))

**TODO:** nhận xét heatmap (các feature kích thước và cân nặng thường tương quan rất cao với nhau, tức có *đa cộng tuyến*, đây chính là lý do PCA hữu ích).

## 10. Tổng kết Phần 1
**TODO:** tóm tắt phát hiện chính và ảnh hưởng của cleaning / integration / transformation.

# Phần 2: PCA

## 11. Áp dụng PCA

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
print("Số thành phần:", X_pca.shape[1])

**TODO:** giải thích tiền xử lý cần thiết (standardize) và cách PC được tính, chọn (như notebook Raisin).

## 12. Explained variance

In [ ]:
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(evr))],
    "explained_variance_ratio": evr.round(4),
    "cumulative": cum.round(4),
})
print(table)

plt.figure(figsize=(7, 4))
plt.bar(range(1, len(evr) + 1), evr)
plt.plot(range(1, len(evr) + 1), cum, marker="o", color="red")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Abalone - explained variance")
plt.tight_layout()
plt.savefig("../figures/abalone_explained_variance.png", dpi=150)
plt.show()

**TODO:** so sánh PC variance cao và thấp. (Gợi ý: ở Abalone PC1 thường chiếm phần rất lớn vì các feature đều đo "độ lớn" của con vật.)

## 13. Thay đổi số thành phần

In [ ]:
for k in range(1, X_scaled.shape[1] + 1):
    p = PCA(n_components=k)
    Xk = p.fit_transform(X_scaled)
    X_back = p.inverse_transform(Xk)
    error = np.mean((X_scaled - X_back) ** 2)
    print(f"k={k}: giữ {p.explained_variance_ratio_.sum():.1%} variance, sai số tái tạo = {error:.4f}")

**TODO:** nhận xét đánh đổi giữa số chiều và thông tin giữ lại.

## 14. Trực quan hóa 2D (PC1 vs PC2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Tô màu theo Sex
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_clean["Sex"].values, alpha=0.5, ax=axes[0])
axes[0].set_title("Màu theo Sex")

# Tô màu theo nhóm tuổi (từ Rings)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=age_group.values, alpha=0.5, ax=axes[1])
axes[1].set_title("Màu theo nhóm tuổi")

for ax in axes:
    ax.set_xlabel(f"PC1 ({evr[0]:.1%})")
    ax.set_ylabel(f"PC2 ({evr[1]:.1%})")

plt.tight_layout()
plt.savefig("../figures/abalone_pca_2d.png", dpi=150)
plt.show()

**Giải thích:** vẽ hai biểu đồ cạnh nhau: một theo `Sex`, một theo nhóm tuổi (chia từ `Rings`).

**TODO:** nhận xét cụm / vùng chồng lấn (thường Infant tách khá rõ, còn M và F chồng nhau nhiều).

In [ ]:
loadings = pd.DataFrame(pca.components_[:2].T, index=feature_cols, columns=["PC1", "PC2"])
loadings.round(3)

## 15. So sánh dữ liệu gốc và dữ liệu PCA
**TODO:** thảo luận ưu điểm (giảm chiều, khử tương quan, trực quan hóa) và hạn chế (mất thông tin, khó diễn giải, chỉ tuyến tính, nhạy outlier/thang đo).